In [18]:
import os
from dotenv import load_dotenv
from langchain_cerebras import ChatCerebras
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from pydantic import BaseModel
import sib_api_v3_sdk
from sib_api_v3_sdk.rest import ApiException


load_dotenv()
os.environ["CEREBRAS_API_KEY"] = os.getenv("CEREBRAS_API_KEY")
os.environ["GROQ_API_KEY"]     = os.getenv("GROQ_API_KEY")
BREVO_KEY = os.getenv("BREVO_API_KEY")
MONGODB_URI=os.getenv("MONGODB_URI")
llm_gpt = ChatCerebras(
    model="gpt-oss-120b",
    api_key=os.getenv("CEREBRAS_API_KEY"),
)
llm_groq = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
)


In [19]:
class User(BaseModel):
    id: int
    name: str
    email: str
    minutes_listened: int
    country: str
    joined_at: str


class EmailDraft(BaseModel):
    recipient: str
    subject: str
    body: str
    reason: str


class CampaignResult(BaseModel):
    total_users: int
    emails_sent: int
    failed: int


In [20]:
from pymongo import MongoClient

client = MongoClient(MONGODB_URI)

db = client["soulsync"]

users_collection = db["users"]

print(users_collection)


Collection(Database(MongoClient(host=['ac-jaaey03-shard-00-00.k7w05ba.mongodb.net:27017', 'ac-jaaey03-shard-00-01.k7w05ba.mongodb.net:27017', 'ac-jaaey03-shard-00-02.k7w05ba.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='SoulSync', authsource='admin', replicaset='atlas-99cwym-shard-0', tls=True), 'soulsync'), 'users')


In [21]:
print(users_collection.count_documents({}))

163


In [22]:
print(users_collection.find_one({}, {"_id": 0}))

{'googleId': '101531911541168397334', '__v': 131, 'createdAt': datetime.datetime(2026, 3, 4, 8, 44, 49, 307000), 'email': 'lokeshvijayraina@gmail.com', 'likedSongs': [{'songId': 'OEfSU98V', 'title': 'Dude – Orchestral Suite', 'artist': 'Sai Abhyankkar', 'albumArt': 'https://c.saavncdn.com/300/Dude-Side-A-Original-Score-Unknown-2025-20251205171547-500x500.jpg', 'duration': 125, 'downloadUrl': [{'quality': '12kbps', 'url': 'https://aac.saavncdn.com/300/8aa77f4cbe46b88fbc4d2998949d933a_12.mp4'}, {'quality': '48kbps', 'url': 'https://aac.saavncdn.com/300/8aa77f4cbe46b88fbc4d2998949d933a_48.mp4'}, {'quality': '96kbps', 'url': 'https://aac.saavncdn.com/300/8aa77f4cbe46b88fbc4d2998949d933a_96.mp4'}, {'quality': '160kbps', 'url': 'https://aac.saavncdn.com/300/8aa77f4cbe46b88fbc4d2998949d933a_160.mp4'}, {'quality': '320kbps', 'url': 'https://aac.saavncdn.com/300/8aa77f4cbe46b88fbc4d2998949d933a_320.mp4'}]}, {'songId': 'd6_r1LCo', 'title': 'God Mode Begins', 'artist': 'Vishnu Edavan, Sai Abhyank

In [23]:
from langchain_core.tools import tool

@tool
def GetTopUsers(limit: int):
    """
    Return the top users ranked by listening time.
    """

    return list(
        users_collection.find(
            {},
            {
                "_id": 0,
                "name": 1,
                "email": 1,
                "totalListeningTime": 1,
            }
        )
        .sort("totalListeningTime", -1)
        .limit(limit)
    )

In [24]:
@tool
def sendMail(
    sender: str,
    receiver: str,
    subject: str,
    body: str,
) -> str:
    """Send an email using Brevo."""

    configuration = sib_api_v3_sdk.Configuration()
    configuration.api_key["api-key"] = BREVO_KEY

    api_instance = sib_api_v3_sdk.TransactionalEmailsApi(
        sib_api_v3_sdk.ApiClient(configuration)
    )

    send_smtp_email = sib_api_v3_sdk.SendSmtpEmail(
        sender={"name": "SoulSync", "email": "lokeshvijayraina@gmail.com"},
        to=[{"email": receiver}],
        subject=subject,
        html_content=body,
    )

    try:
        api_response = api_instance.send_transac_email(send_smtp_email)
        print("Success! Message ID:", api_response.message_id)
        print(sender, receiver, subject)
        return f"Mail sent successfully to {receiver}"
    except ApiException as e:
        print(" Error:", e)
        return f"Failed to send email: {e}"


In [27]:
agent = create_agent(
    model=llm_gpt,
    tools=[GetTopUsers, sendMail],
    system_prompt="""
You are LOOKOUT.

You have two tools:

1. GetTopUsers(limit)  – returns the highest-ranked users.
2. sendMail(sender, receiver, subject, body)  – sends an email.

Workflow:
1. Find the top users.
2. Write a personalised thank-you email for each.
3. Send each email.
4. Repeat until every user has received one.
"""
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="""
Reward our top 5 users.

Sender: lokeshvijayraina@gmail.com

Write a professional thank-you email for each user and send it.
"""
            )
        ]
    }
)


Success! Message ID: <202607051806.34536681035@smtp-relay.mailin.fr>
lokeshvijayraina@gmail.com lokeshvijayraina@gmail.com Thank You for Being a Top Listener!
Success! Message ID: <202607051806.95564868698@smtp-relay.mailin.fr>
lokeshvijayraina@gmail.com ajaysrinatha59@gmail.com Thank You for Being a Top Listener!
Success! Message ID: <202607051806.92154509056@smtp-relay.mailin.fr>
lokeshvijayraina@gmail.com weortheboys1513@gmail.com Thank You for Being a Top Listener!
Success! Message ID: <202607051806.78483681476@smtp-relay.mailin.fr>
lokeshvijayraina@gmail.com sankarelumalai6@gmail.com Thank You for Being a Top Listener!
Success! Message ID: <202607051807.22054492438@smtp-relay.mailin.fr>
lokeshvijayraina@gmail.com jaibharath04102005@gmail.com Thank You for Being a Top Listener!
